In [1]:
import re
import pandas as pd

df = pd.read_csv("../data/processed/netflix_clean.csv", parse_dates=["date_added"])

def to_tokens(value):
    if value == "Unknown":
        return ""
    parts = [re.sub(r"[^\w]", "", p) for p in value.split(",")]
    return " ".join(parts)

df["features"] = (
    df["type"].apply(to_tokens) + " "
    + df["rating"].apply(to_tokens) + " "
    + df["listed_in"].apply(to_tokens) + " "
    + df["director"].apply(to_tokens) + " "
    + df["country"].apply(to_tokens)
).str.lower().str.split().str.join(" ")

In [2]:
print(df.shape)
print(df.loc[[0, 1, 2], ["title", "features"]].to_string())
print((df["features"].str.len() == 0).sum(), df["features"].str.split().str.len().describe()[["min", "mean", "max"]])

(8786, 18)
                  title                                                                               features
0  Dick Johnson Is Dead                                   movie pg13 documentaries kirstenjohnson unitedstates
1             Ganglands  tvshow tvma crimetvshows internationaltvshows tvactionadventure julienleclercq france
2         Midnight Mass                    tvshow tvma tvdramas tvhorror tvmysteries mikeflanagan unitedstates
0 min      4.000000
mean     5.961074
max     18.000000
Name: features, dtype: float64


In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

tfidf = TfidfVectorizer(token_pattern=r"\S+")
matrix = tfidf.fit_transform(df["features"])
print(matrix.shape, len(tfidf.vocabulary_))

(8786, 5127) 5127


In [4]:
def recommend(title, n=5):
    matches = df.index[df["title"].str.lower() == title.lower()]
    if len(matches) == 0:
        return "Title not found"
    i = matches[0]
    scores = linear_kernel(matrix[i], matrix).ravel()
    picks = []
    for j in scores.argsort()[::-1]:
        if df.loc[j, "title"].lower() == title.lower():
            continue
        picks.append(j)
        if len(picks) == n:
            break
    result = df.loc[picks, ["title", "type", "rating", "listed_in", "director"]].copy()
    result["similarity"] = scores[picks].round(3)
    return result

recommend("Dick Johnson Is Dead")

,title,type,rating,listed_in,director,similarity
7436,The Irishman: In Conversation,Movie,TV-MA,Documentaries,Unknown,0.336
8045,Liberated: The New Sexual Revolution,Movie,TV-MA,Documentaries,Unknown,0.336
7264,Murder to Mercy: The Cyntoia Brown Story,Movie,TV-MA,Documentaries,Unknown,0.336
8309,Virunga: Gorillas in Peril,Movie,TV-14,Documentaries,Unknown,0.323
7288,Surviving R. Kelly: The Impact,Movie,TV-14,Documentaries,Unknown,0.323
